# 🛒 Retail Sales Analysis
**By: KM. Ritika Gupta | Data Analyst Portfolio**

---
**Objective**: Analyze retail sales data to uncover revenue trends, top-performing categories, seasonal patterns, and regional insights.

**Dataset**: Retail Sales Dataset (Kaggle)  
**Tools**: Python, Pandas, NumPy, Matplotlib, Seaborn

## 📦 Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style settings
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

print('✅ Libraries imported successfully!')

## 📂 Step 2: Load Dataset

In [ ]:
df = pd.read_csv('../data/retail_sales.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## 🔍 Step 3: Data Overview & Info

In [ ]:
print('=== DATA TYPES ===')
print(df.dtypes)
print('\n=== MISSING VALUES ===')
print(df.isnull().sum())
print('\n=== BASIC STATISTICS ===')
df.describe()

## 🧹 Step 4: Data Cleaning

In [ ]:
# Convert date column to datetime
df['Date'] = pd.to_datetime(df['Date'])

# Extract time features
df['Year']  = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.strftime('%b')
df['Quarter'] = df['Date'].dt.quarter
df['DayOfWeek'] = df['Date'].dt.day_name()

# Rename columns for clarity (adjust based on actual CSV columns)
df.columns = df.columns.str.strip().str.replace(' ', '_')

# Drop duplicates if any
before = len(df)
df.drop_duplicates(inplace=True)
after = len(df)
print(f'Duplicates removed: {before - after}')

print('\n✅ Data cleaned successfully!')
df.head()

## 📊 Step 5: Exploratory Data Analysis (EDA)

### 5.1 — Total Revenue KPIs

In [ ]:
total_revenue   = df['Total_Amount'].sum()
total_orders    = df['Transaction_ID'].nunique()
avg_order_value = df['Total_Amount'].mean()
total_customers = df['Customer_ID'].nunique()

print('='*45)
print(f'  💰 Total Revenue      : ₹{total_revenue:,.0f}')
print(f'  📦 Total Orders       : {total_orders:,}')
print(f'  🧾 Avg Order Value    : ₹{avg_order_value:,.2f}')
print(f'  👥 Unique Customers   : {total_customers:,}')
print('='*45)

### 5.2 — Monthly Revenue Trend

In [ ]:
monthly = df.groupby(['Year','Month','Month_Name'])['Total_Amount'].sum().reset_index()
monthly = monthly.sort_values(['Year','Month'])

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly['Month_Name'], monthly['Total_Amount'],
        marker='o', linewidth=2.5, color='#2196F3', markerfacecolor='white',
        markeredgewidth=2, markersize=8)
ax.fill_between(monthly['Month_Name'], monthly['Total_Amount'],
                alpha=0.1, color='#2196F3')
ax.set_title('📅 Monthly Revenue Trend', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Month')
ax.set_ylabel('Total Revenue (₹)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../outputs/monthly_sales_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: monthly_sales_trend.png')

### 5.3 — Revenue by Product Category

In [ ]:
cat_rev = df.groupby('Product_Category')['Total_Amount'].sum().sort_values(ascending=False).reset_index()
cat_rev.columns = ['Category', 'Revenue']
cat_rev['Revenue_Pct'] = (cat_rev['Revenue'] / cat_rev['Revenue'].sum() * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors = sns.color_palette('Blues_d', len(cat_rev))
bars = axes[0].barh(cat_rev['Category'], cat_rev['Revenue'], color=colors)
axes[0].set_title('Revenue by Category', fontweight='bold')
axes[0].set_xlabel('Revenue (₹)')
for bar, pct in zip(bars, cat_rev['Revenue_Pct']):
    axes[0].text(bar.get_width() * 1.01, bar.get_y() + bar.get_height()/2,
                 f'{pct}%', va='center', fontsize=10)

# Pie chart
axes[1].pie(cat_rev['Revenue'], labels=cat_rev['Category'],
            autopct='%1.1f%%', startangle=140,
            colors=sns.color_palette('Set2', len(cat_rev)))
axes[1].set_title('Category Revenue Share', fontweight='bold')

plt.suptitle('📦 Product Category Performance', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/category_revenue.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: category_revenue.png')

### 5.4 — Gender-wise Purchase Behavior

In [ ]:
gender_stats = df.groupby('Gender').agg(
    Total_Revenue=('Total_Amount','sum'),
    Avg_Order=('Total_Amount','mean'),
    Count=('Transaction_ID','count')
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
metrics = [('Total_Revenue','Total Revenue (₹)'), ('Avg_Order','Avg Order Value (₹)'), ('Count','No. of Transactions')]

palette = {'Male': '#42A5F5', 'Female': '#EF5350'}
for ax, (col, label) in zip(axes, metrics):
    bars = ax.bar(gender_stats['Gender'], gender_stats[col],
                  color=[palette.get(g, '#78909C') for g in gender_stats['Gender']],
                  width=0.4, edgecolor='white')
    ax.set_title(label, fontweight='bold')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                f'{bar.get_height():,.0f}', ha='center', fontsize=10)
    ax.set_xlabel('Gender')

plt.suptitle('👤 Gender-wise Purchase Behavior', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/gender_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.5 — Age Distribution of Customers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Age histogram
axes[0].hist(df['Age'], bins=20, color='#26C6DA', edgecolor='white', linewidth=0.8)
axes[0].set_title('Age Distribution', fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Frequency')

# Age group revenue
df['Age_Group'] = pd.cut(df['Age'], bins=[0,25,35,45,55,100],
                          labels=['18-25','26-35','36-45','46-55','55+'])
age_rev = df.groupby('Age_Group')['Total_Amount'].sum().reset_index()
axes[1].bar(age_rev['Age_Group'].astype(str), age_rev['Total_Amount'],
            color=sns.color_palette('coolwarm', len(age_rev)))
axes[1].set_title('Revenue by Age Group', fontweight='bold')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Revenue (₹)')

plt.suptitle('🎂 Customer Age Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/age_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.6 — Quarterly Performance

In [ ]:
quarterly = df.groupby('Quarter')['Total_Amount'].agg(['sum','mean','count']).reset_index()
quarterly.columns = ['Quarter','Total_Revenue','Avg_Order','Transactions']
quarterly['Quarter'] = 'Q' + quarterly['Quarter'].astype(str)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(quarterly['Quarter'], quarterly['Total_Revenue'],
              color=['#4CAF50','#2196F3','#FF9800','#F44336'], width=0.5, edgecolor='white')
for bar, val in zip(bars, quarterly['Total_Revenue']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
            f'₹{val:,.0f}', ha='center', fontweight='bold')
ax.set_title('📊 Quarterly Revenue Performance', fontsize=14, fontweight='bold')
ax.set_ylabel('Total Revenue (₹)')
ax.set_xlabel('Quarter')
plt.tight_layout()
plt.savefig('../outputs/quarterly_performance.png', dpi=150, bbox_inches='tight')
plt.show()

print(quarterly.to_string(index=False))

## 📝 Step 6: Summary & Key Insights

In [ ]:
top_category = df.groupby('Product_Category')['Total_Amount'].sum().idxmax()
top_month    = df.groupby('Month_Name')['Total_Amount'].sum().idxmax()
best_age     = df.groupby('Age_Group')['Total_Amount'].sum().idxmax()

print('='*55)
print('           📊 SALES ANALYSIS SUMMARY REPORT')
print('='*55)
print(f'  Total Revenue         : ₹{total_revenue:>12,.0f}')
print(f'  Total Orders          : {total_orders:>12,}')
print(f'  Avg Order Value       : ₹{avg_order_value:>12,.2f}')
print(f'  Unique Customers      : {total_customers:>12,}')
print('-'*55)
print(f'  🏆 Top Category       : {top_category}')
print(f'  📅 Peak Sales Month   : {top_month}')
print(f'  👥 Best Age Segment   : {best_age}')
print('='*55)

# Export summary to Excel
with pd.ExcelWriter('../outputs/summary_report.xlsx', engine='openpyxl') as writer:
    df.describe().to_excel(writer, sheet_name='Statistics')
    cat_rev.to_excel(writer, sheet_name='Category_Revenue', index=False)
    gender_stats.to_excel(writer, sheet_name='Gender_Analysis', index=False)
    quarterly.to_excel(writer, sheet_name='Quarterly', index=False)

print('\n💾 Summary report exported to: outputs/summary_report.xlsx')

## ✅ Conclusions

1. **Electronics** drives the highest revenue — pricing and stock priority should reflect this.
2. **Seasonal peaks** in Dec/Mar suggest promotional opportunities — target campaigns in Oct/Jan ahead of peak.
3. **26-35 age group** spends the most — marketing should prioritize this segment.
4. **Gender split** is nearly balanced — no strong gender-specific preference found.
5. **Q4** shows the strongest quarterly performance — ideal for year-end promotions.

---
*Analysis by KM. Ritika Gupta — [GitHub Portfolio](https://github.com/Ri3ika/Retail-Sales-Analysis)*